<a href="https://colab.research.google.com/github/kevinyaelvillalpandomartinez-tech/Diagn-stico-estrat-gico-integral-para-RappiPlus/blob/main/Kevin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np

import seaborn as sns

import matplotlib.pyplot as plt


In [ ]:
# cargar archivos
orders =pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv") # tu código aquí
catalog =pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv") # tu código aquí
marketing =pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv") # tu código aquí

In [ ]:
# explorar datasets
orders.info()
# tu código aquí

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [ ]:
catalog.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [ ]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# Validar y convertir fechas al formato correcto
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"])
marketing["fecha"] = pd.to_datetime(marketing["fecha"])



In [ ]:
#Revisar variables numéricas (sin negativos o ceros inválidos)
print("cantidad negativos en la columna cantidad",(orders["cantidad"] < 0).sum())
print("cantidad de 0 en la columna cantidad",(orders["cantidad"] == 0).sum())
print("cantidad de nulos  en la columna cantidad",orders["cantidad"].isnull().sum())
print("\n")

print("cantidad negativos  en la columna precio_unitario",(orders["precio_unitario"] < 0).sum())
print("cantidad 0  en la columna precio_unitario",(orders["precio_unitario"] == 0).sum())
print("cantidad de nulos  en la columna precio_unitario",orders["precio_unitario"].isnull().sum())
print("\n")


print("cantidad negativos en la columna 'monto_total'",(orders['monto_total'] < 0).sum())
print("cantidad de 0 en la columna 'monto_total'",(orders['monto_total'] == 0).sum())
print("cantidad de nulos  en la columna 'monto_total'",orders['monto_total'].isnull().sum())
print("\n")

print("cantidad negativos en la columna 'monto_descuento'",(orders['monto_descuento'] < 0).sum())
print("cantidad  ceros invalidos en la columna 'monto_total'",(orders['monto_descuento'] == 0).sum())
print("cantidad de nulos  en la columna 'monto_descuento'",orders['monto_descuento'].isnull().sum())


orders[['cantidad', 'precio_unitario', 'monto_total', 'monto_descuento']].describe()

cantidad negativos en la columna cantidad 4
cantidad de 0 en la columna cantidad 0
cantidad de nulos  en la columna cantidad 50


cantidad negativos  en la columna precio_unitario 0
cantidad 0  en la columna precio_unitario 0
cantidad de nulos  en la columna precio_unitario 50


cantidad negativos en la columna 'monto_total' 4
cantidad de 0 en la columna 'monto_total' 0
cantidad de nulos  en la columna 'monto_total' 0


cantidad negativos en la columna 'monto_descuento' 0
cantidad  ceros invalidos en la columna 'monto_total' 12551
cantidad de nulos  en la columna 'monto_descuento' 50


,cantidad,precio_unitario,monto_total,monto_descuento
count,25050.000000,25050.000000,2.510000e+04,25050.000000
mean,7.092735,259.305549,2.072680e+03,4.500798
std,296.277003,138.726461,9.894995e+04,5.223010
min,-2.000000,20.030000,-4.926500e+02,0.000000
25%,1.000000,138.377500,1.805075e+02,0.000000
50%,2.000000,258.715000,3.417500e+02,0.000000
75%,2.000000,380.332500,5.185800e+02,10.000000
max,20000.000000,499.960000,8.840200e+06,15.000000


In [ ]:
#no tiene numeros nehativos en la columna de gasto
print("cantidad negativos en la columna 'gasto'",(marketing['gasto'] < 0).sum())
print("cantidad de 0 en la columna 'gasto'",(marketing['gasto'] == 0).sum())
print("cantidad de nulos  en la columna 'gasto'",marketing['gasto'].isnull().sum())
marketing.describe()


cantidad negativos en la columna 'gasto' 0
cantidad de 0 en la columna 'gasto' 0
cantidad de nulos  en la columna 'gasto' 0


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [ ]:

orders = orders[(orders["cantidad"] >= 0) & (orders["monto_total"] >= 0)]
#decidimos elimminar los valores negativos porque representan un porcion muy pequeÑa del dataset
print("cantidad negativos en la columna cantidad",(orders["cantidad"] < 0).sum())
print("cantidad de 0 en la columna cantidad",(orders["cantidad"] == 0).sum())
print("cantidad de nulos  en la columna cantidad",orders["cantidad"].isnull().sum())
print("\n")
print("cantidad negativos en la columna 'monto_total'",(orders['monto_total'] < 0).sum())
print("cantidad de 0 en la columna 'monto_total'",(orders['monto_total'] == 0).sum())
print("cantidad de nulos  en la columna 'monto_total'",orders['monto_total'].isnull().sum())
print("\n")
orders.describe()

cantidad negativos en la columna cantidad 0
cantidad de 0 en la columna cantidad 0
cantidad de nulos  en la columna cantidad 0


cantidad negativos en la columna 'monto_total' 0
cantidad de 0 en la columna 'monto_total' 0
cantidad de nulos  en la columna 'monto_total' 0




,cantidad,precio_unitario,monto_descuento,monto_total
count,25046.000000,25046.000000,25046.000000,2.504600e+04
mean,7.094067,259.304400,4.500719,2.076331e+03
std,296.300643,138.715188,5.223232,9.905653e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.405000,0.000000,1.804025e+02
50%,2.000000,258.715000,0.000000,3.415600e+02
75%,2.000000,380.272500,10.000000,5.184375e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [ ]:
#Verificar consistencia de montos
orders['monto_total_esperado'] = (
    orders['precio_unitario'] *
    orders['cantidad']
) - orders['monto_descuento']

orders['diferencia'] = (
    orders['monto_total'] -
    orders['monto_total_esperado']
).abs()

inconsistencias = (orders['diferencia'] > 0.01).sum()

print(f"Inconsistencias encontradas: {inconsistencias}")

Inconsistencias encontradas: 1146


In [ ]:
orders["diferencia"].describe()
orders[['monto_total',
        'monto_total_esperado',
        'diferencia']].sort_values(
            by='diferencia',
            ascending=False
        ).head(20)

,monto_total,monto_total_esperado,diferencia
9721,849.07,849.06,0.01
17851,975.07,975.06,0.01
19297,712.43,712.44,0.01
10655,652.07,652.06,0.01
8501,554.69,554.68,0.01
17025,528.43,528.44,0.01
3309,825.07,825.06,0.01
13470,772.19,772.18,0.01
11348,739.31,739.32,0.01
23297,671.93,671.94,0.01


In [ ]:
Se identificaron 1,146 registros con diferencias entre el monto total registrado y el monto total calculado a partir de las variables transaccionales, lo que representa aproximadamente el 4.6% de los pedidos. Antes de aplicar correcciones, se analizará la magnitud de estas discrepancias para determinar si corresponden a errores de captura, diferencias por redondeo o reglas de negocio adicionales no contempladas en la fórmula utilizada. Por el momento, los registros se mantienen en el dataset y el hallazgo queda documentado para su consideración en análisis posteriores.

SyntaxError: invalid syntax (3728331710.py, line 1)

In [ ]:
#Eliminar duplicados
#verificamos si existen duplicados
print(orders.duplicated().sum())

orders.duplicated(subset=['id_pedido']).sum()

#Eliminamos
orders = orders.drop_duplicates(subset=['id_pedido'], keep='first')

# verificamos que se borraron los duplicados
print("verificacion de duplicados despues de limpieza del datast orders",orders.duplicated(subset=['id_pedido']).sum())


#verificamos dataset marketing
print("los duplicados de el dataset son marketing " , marketing.duplicated(subset=['id_campaña',"fecha"]).sum())

100
verificacion de duplicados despues de limpieza del datast orders 0
los duplicados de el dataset son marketing  0


In [ ]:
#revisamos variables categoricas

#dataset orders
columnas_categoricas_orders = orders[["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]]
for col in columnas_categoricas_orders:
 print(columnas_categoricas_orders[col].value_counts())



Colombia     7467
Mexico       7465
Argentina    7239
mexico        862
colombia      822
argentina     795
Name: pais, dtype: int64
desktop    12681
mobile     12245
Name: dispositivo, dtype: int64
social         8385
organic        8269
paid_search    8262
Name: fuente_referencia, dtype: int64
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
Name: nombre_producto, dtype: int64
Hogar          8346
Moda           8295
Electronica    8275
Name: categoria_producto, dtype: int64


In [ ]:
#corregimos la columnas
orders["pais"] = orders["pais"].str.title()
columnas_categoricas_orders = orders[["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]]
for col in columnas_categoricas_orders:
 print(columnas_categoricas_orders[col].value_counts())


Mexico       8327
Colombia     8289
Argentina    8034
Name: pais, dtype: int64
desktop    12681
mobile     12245
Name: dispositivo, dtype: int64
social         8385
organic        8269
paid_search    8262
Name: fuente_referencia, dtype: int64
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
Name: nombre_producto, dtype: int64
Hogar          8346
Moda           8295
Electronica    8275
Name: categoria_producto, dtype: int64


In [ ]:
#dataset catalogs
columnas_categoricas_catalog = catalog[["nombre_producto","categoria_producto","proveedor"]]
for col in columnas_categoricas_catalog:
 print(columnas_categoricas_catalog[col].value_counts())

Jacket-Winter-M         1
Phone-Pro-128GB         1
Laptop-Gaming-16GB      1
Vacuum-Pro-Black        1
Blender-XL-Red          1
Tablet-Standard-64GB    1
Sneakers-Urban-42       1
Name: nombre_producto, dtype: int64
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64
Fuller, Pena and Myers     1
Long-Reid                  1
Mcmillan-Rhodes            1
Rivera, Carr and Finley    1
Bowers LLC                 1
Greene-Smith               1
King Ltd                   1
Name: proveedor, dtype: int64


In [ ]:
#dataset marketing
columnas_categoricas_marketing = marketing[["pais","canal"]]
for col in columnas_categoricas_marketing:
 print(columnas_categoricas_marketing[col].value_counts())

Mexico       540
Argentina    540
Colombia     540
Name: pais, dtype: int64
paid_search    507
organic        506
social         506
Name: canal, dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:

# tu código aquí
#¿Cuál es el ingreso total (revenue)
ingreso_total  =  orders["monto_total"].sum()
print(f"ingreso total: ${ingreso_total:,.2f}")
print("\n")
#¿Cuál es el costo total?
orders_catalog = orders.merge(
    catalog,
    on = "nombre_producto",
    how="left"
)
orders_catalog["costo_total"] = (
    orders_catalog["cantidad"] *
    orders_catalog["costo_unitario"]
)
costo_total = orders_catalog["costo_total"].sum()
print(f"Costo Total: ${costo_total:,.2f}")
print("\n")


#¿Cuánto se ha invertido en marketing?
costo_total_marketing = marketing["gasto"].sum()
print(f"gasto_total_marketing: ${costo_total_marketing:,.2f}")
print("\n")

#¿El negocio es rentable?
utilidad = ingreso_total - costo_total - costo_total_marketing
print(f"utilidad del negocio: ${utilidad:,.2f}")

ingreso total: $51,966,981.56


Costo Total: $43,124,069.01


gasto_total_marketing: $2,871,843.53


utilidad del negocio: $5,971,069.02


In [ ]:
#¿cual es el ticket promedio por orden?
ticket_promedio =  orders["monto_total"].mean()
print(f"el ticket promedio es: ${ticket_promedio:,.2f}")
print("\n")

#¿Cuál es la cantidad promedio de productos por orden?
cantidad_promedio_productos = orders["cantidad"].mean()
print(f"la cantidad promedio de ordenes es: {cantidad_promedio_productos:,.2f}")
print("\n")
#¿cual es el producto mas vendido?
producto_mas_vendido = (
    orders.groupby("nombre_producto")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)

print(producto_mas_vendido.head())
print("\n")


#Cuánto se ha gastado en marketing por canal?
marketing_por_canal = (
    marketing.groupby("canal")["gasto"]
    .sum()
    .sort_values(ascending=False)
)

print(marketing_por_canal)


el ticket promedio es: $2,083.18


la cantidad promedio de ordenes es: 7.12


nombre_producto
Laptop-Gaming-16GB    144198.0
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Name: cantidad, dtype: float64


canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================


query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'add_to_cart' THEN 2
            WHEN nombre_evento = 'select_item' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS paso
    FROM events
    GROUP BY nombre_evento
)

SELECT
    nombre_evento,
    usuarios,
    ROUND(
        usuarios * 100.0 /
        LAG(usuarios) OVER (ORDER BY paso),
        2
    ) AS conversion_pct
FROM funnel
ORDER BY paso;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,nombre_evento,usuarios,conversion_pct
0,first_visit,7796,NaN
1,add_to_cart,7634,97.92
2,select_item,7582,99.32
3,begin_checkout,7208,95.07
4,add_payment_info,6250,86.71
5,purchase,6240,99.84


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''

SELECT * FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
# ======================
query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes,
        CAST(fecha_registro AS DATE) AS fecha_registro
    FROM users
),

-- Tamaño real de cada cohorte
cohort_size AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM cohortes
    GROUP BY cohorte_mes
),

-- Solo usuarios realmente activos
actividad AS (
    SELECT
        c.id_usuario,
        c.cohorte_mes,
        FLOOR(a.dias_despues_registro / 7.0) AS semana
    FROM cohortes c
    JOIN user_activity a
        ON c.id_usuario = a.id_usuario
    WHERE a.activo = 1
),

retencion AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT CASE WHEN semana = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN semana = 2 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN semana = 3 THEN id_usuario END) AS retenido_w3
    FROM actividad
    GROUP BY cohorte_mes
)

SELECT
    cs.cohorte_mes,
    cs.usuarios_iniciales,

    COALESCE(r.retenido_w1, 0) AS retenido_w1,
    COALESCE(r.retenido_w2, 0) AS retenido_w2,
    COALESCE(r.retenido_w3, 0) AS retenido_w3,

    ROUND(
        COALESCE(r.retenido_w1, 0) * 100.0 /
        cs.usuarios_iniciales, 2
    ) AS semana_1,

    ROUND(
        COALESCE(r.retenido_w2, 0) * 100.0 /
        cs.usuarios_iniciales, 2
    ) AS semana_2,

    ROUND(
        COALESCE(r.retenido_w3, 0) * 100.0 /
        cs.usuarios_iniciales, 2
    ) AS semana_3

FROM cohort_size cs
LEFT JOIN retencion r
    ON cs.cohorte_mes = r.cohorte_mes

ORDER BY cs.cohorte_mes;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)

cohorte_final

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No existe diferencia estadísticamente significativa entre las tasas de conversión de ambos grupos.
   - **H₁ (Hipótesis alternativa):** Existe una diferencia estadísticamente significativa entre las tasas de conversión de ambos grupos.
   
**Test estadístico:**
Dado que la variable conversion es binaria (0 o 1), el test más adecuado es:

Test Z para diferencia de proporciones

Este test permite comparar si la diferencia observada entre dos tasas de conversión puede atribuirse al azar o si es suficientemente grande para considerarse un efecto real.
**Nivel de significancia alpha:** α=0.05

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
experiment =pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv') # tu código aquí
experiment.info()

experiment["variante"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


tratamiento    5035
control        4965
Name: variante, dtype: int64

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Grupo control
control = experiment.loc[
    experiment["variante"] == "control",
    "convirtio"
]

# Grupo tratamiento
tratamiento = experiment.loc[
    experiment["variante"] == "tratamiento",
    "convirtio"
]

# Número de conversiones
conversiones = [
    control.sum(),
    tratamiento.sum()
]

# Número de usuarios por grupo
usuarios = [
    len(control),
    len(tratamiento)
]

# Tasas de conversión
conversion_control = control.mean() * 100
conversion_tratamiento = tratamiento.mean() * 100

print(f"Conversión Control: {conversion_control:.2f}%")
print(f"Conversión Tratamiento: {conversion_tratamiento:.2f}%")

# Test Z para diferencia de proporciones
z_stat, p_value = proportions_ztest(
    count=conversiones,
    nobs=usuarios
)

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpretación
alpha = 0.05

if p_value < alpha:
    print("\nSe rechaza H₀.")
    print("Existe evidencia estadística de que la nueva UI del checkout impacta la conversión.")
else:
    print("\nNo se rechaza H₀.")
    print("No existe evidencia suficiente para afirmar que la nueva UI impacta la conversión.")

Conversión Control: 15.69%
Conversión Tratamiento: 16.29%

Z-statistic: -0.8133
P-value: 0.4161

No se rechaza H₀.
No existe evidencia suficiente para afirmar que la nueva UI impacta la conversión.


In [ ]:
Interpretacion del resultado
Interpretación Estadística

El valor p-value = 0.4161 es considerablemente mayor que el nivel de significancia establecido (α = 0.05).

Por lo tanto:

No se rechaza la hipótesis nula (H₀).

Esto significa que no existe evidencia estadística suficiente para concluir que la nueva versión del checkout haya generado un cambio significativo en la tasa de conversión.

📈 Diferencia de Conversión

La versión de tratamiento obtuvo una tasa de conversión de 16.29%, mientras que el grupo control obtuvo 15.69%.

La diferencia observada es:

16.29%−15.69%=0.60%

16.29%−15.69%=0.60%

Aunque el tratamiento muestra una conversión ligeramente superior, la diferencia es pequeña y podría explicarse por variabilidad aleatoria de la muestra.

📉 Interpretación del Z-Statistic

El estadístico Z obtenido fue -0.8133.

Un valor cercano a cero indica que las proporciones de conversión de ambos grupos son muy similares y que la diferencia observada es pequeña en relación con la variabilidad esperada.

Esto refuerza la conclusión de que el cambio en la interfaz no produjo un efecto estadísticamente detectable.

🎯 Conclusión de Negocio

La nueva UI del checkout mostró una mejora marginal de 0.6 puntos porcentuales en la conversión; sin embargo, el test estadístico indica que dicha diferencia no es significativa.

Por lo tanto:

No existe evidencia suficiente para justificar la implementación del nuevo diseño basándose únicamente en la conversión.
Se recomienda mantener la versión actual o realizar nuevos experimentos con cambios más relevantes.
También puede ser útil analizar métricas complementarias como tiempo en checkout, abandono del carrito, duración de sesión o conversión por dispositivo para identificar posibles beneficios indirectos del nuevo diseño.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive
https://drive.google.com/drive/folders/1IrZw9ITitjFFZkt9YKqf3-Ov3pv4tenF?usp=sharing